In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# BlindDetection-V1 N_dev=256 calibration handoff

Prepared only. Select a GPU runtime and run the cells once after the reviewed producer exact is available on GitHub. The science denominator remains zero.

In [ ]:
from google.colab import userdata
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys
import torch
import zipfile

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
PRODUCER_EXACT = '686b21e7e7d0358ca5a7dff78ac781b538087e35'
CHECKOUT = Path('/content/cegwm-blind-detection-v1-calibration')
RUN_ID = f"blind-detection-v1-{PRODUCER_EXACT}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
LOCAL_ROOT = Path('/content') / (RUN_ID + '-local')
RUNTIME_ROOT = LOCAL_ROOT / 'runtime'
LOCAL_RESULT = LOCAL_ROOT / 'calibration_result.json'
LOCAL_ROWS = LOCAL_ROOT / 'calibration_rows.json'
LOCAL_REPLAY = LOCAL_ROOT / 'fresh_replay_rows.json'
LOCAL_THRESHOLD = LOCAL_ROOT / 'blind_detection_v1_thresholds.json'
LOCAL_STDOUT = LOCAL_ROOT / 'runner.stdout.txt'
LOCAL_STDERR = LOCAL_ROOT / 'runner.stderr.txt'
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/calibration-runs')
TERMINAL_ZIP = DRIVE_ROOT / f'{RUN_ID}.zip'

if 'BLIND_CALIBRATION_RUNNER_CALLS' not in globals():
    BLIND_CALIBRATION_RUNNER_CALLS = 0
if 'BLIND_TERMINAL_ZIP_WRITES' not in globals():
    BLIND_TERMINAL_ZIP_WRITES = 0
if BLIND_CALIBRATION_RUNNER_CALLS != 0 or BLIND_TERMINAL_ZIP_WRITES != 0:
    raise RuntimeError('this notebook execution cell is single-use')
if CHECKOUT.exists() or LOCAL_ROOT.exists() or TERMINAL_ZIP.exists():
    raise FileExistsError('fresh checkout, local root, and terminal ZIP path required')
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
with LOCAL_STDOUT.open('xb'):
    pass
with LOCAL_STDERR.open('xb'):
    pass

def run_logged(command, *, cwd=None, env=None, check=True):
    with LOCAL_STDOUT.open('ab') as stdout, LOCAL_STDERR.open('ab') as stderr:
        return subprocess.run(
            command, cwd=cwd, env=env, stdout=stdout, stderr=stderr, check=check,
        )

def write_notebook_failure(stage, error):
    if LOCAL_RESULT.exists():
        return
    payload = {
        'calibration_rows': [],
        'claim_ceiling': 'engineering_N_dev_256_threshold_calibration_only; science_denominator=0',
        'denominator': 256,
        'error': f'{type(error).__name__}: {error}',
        'fresh_replay_rows': [],
        'producer_exact': PRODUCER_EXACT,
        'science_denominator': 0,
        'stage': stage,
        'status': 'OPERATIONAL_BLOCKED',
        'threshold_candidate_ready': False,
    }
    with LOCAL_RESULT.open('xb') as sink:
        sink.write(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('ascii'))

def write_terminal_zip(success):
    global BLIND_TERMINAL_ZIP_WRITES
    if BLIND_TERMINAL_ZIP_WRITES != 0:
        raise RuntimeError('terminal ZIP publication may be attempted only once')
    BLIND_TERMINAL_ZIP_WRITES += 1
    result = json.loads(LOCAL_RESULT.read_text(encoding='ascii'))
    with LOCAL_ROWS.open('xb') as sink:
        sink.write(json.dumps(result.get('calibration_rows', []), sort_keys=True, separators=(',', ':')).encode('ascii'))
    with LOCAL_REPLAY.open('xb') as sink:
        sink.write(json.dumps(result.get('fresh_replay_rows', []), sort_keys=True, separators=(',', ':')).encode('ascii'))
    members = [LOCAL_RESULT, LOCAL_ROWS, LOCAL_REPLAY, LOCAL_STDOUT, LOCAL_STDERR]
    if success:
        members.append(LOCAL_THRESHOLD)
    with zipfile.ZipFile(TERMINAL_ZIP, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
        for member in members:
            archive.write(member, arcname=member.name)

stage = 'environment_guard'
runner_env = None
completed = None
terminal_published = False
root_key = ''
hf_token = ''
try:
    if not torch.cuda.is_available():
        raise RuntimeError('GPU required; N_dev=256 was not executed')
    stage = 'detached_checkout'
    run_logged(['git', 'clone', REPO_URL, str(CHECKOUT)])
    run_logged(['git', '-C', str(CHECKOUT), 'checkout', '--detach', PRODUCER_EXACT])
    def git(*args):
        return subprocess.run(
            ['git', '-C', str(CHECKOUT), *args], check=True, capture_output=True, text=True,
        ).stdout.strip()
    if git('rev-parse', 'HEAD') != PRODUCER_EXACT or git('branch', '--show-current') != '':
        raise RuntimeError('detached producer exact differs')
    if git('status', '--porcelain=v1') != '':
        raise RuntimeError('producer checkout must be clean')
    stage = 'dependency_install'
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', str(CHECKOUT)])
    if git('rev-parse', 'HEAD') != PRODUCER_EXACT or git('status', '--porcelain=v1') != '':
        raise RuntimeError('producer exact or clean state changed during installation')
    stage = 'import_validation'
    run_logged([
        sys.executable, '-c',
        'from experiments import run_blind_detection_v1 as r; r.load_roster_inputs(r.REPO_ROOT); r.load_runtime_config(r.REPO_ROOT)',
    ], cwd=CHECKOUT)
    stage = 'secret_validation'
    root_key = userdata.get('CEG_WM_ROOT_KEY')
    hf_token = userdata.get('HF_TOKEN')
    if not isinstance(root_key, str) or not root_key.strip():
        raise RuntimeError('CEG_WM_ROOT_KEY Colab Secret is required')
    if not isinstance(hf_token, str) or not hf_token.strip():
        raise RuntimeError('HF_TOKEN Colab Secret is required')
    secret_markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
    runner_env = {
        name: value for name, value in os.environ.items()
        if not any(marker in name.upper() for marker in secret_markers)
    }
    runner_env['CEG_WM_ROOT_KEY'] = root_key
    runner_env['HF_TOKEN'] = hf_token
    root_key = ''
    hf_token = ''
    stage = 'formal_runner'
    command = [
        sys.executable, str(CHECKOUT / 'experiments/run_blind_detection_v1.py'),
        'calibrate-and-freeze', '--producer-exact', PRODUCER_EXACT,
        '--runtime-root', str(RUNTIME_ROOT),
        '--candidate-output', str(LOCAL_THRESHOLD),
        '--result-output', str(LOCAL_RESULT),
    ]
    BLIND_CALIBRATION_RUNNER_CALLS += 1
    completed = run_logged(command, cwd=CHECKOUT, env=runner_env, check=False)
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    runner_env = None
    if BLIND_CALIBRATION_RUNNER_CALLS != 1 or not LOCAL_RESULT.is_file():
        raise RuntimeError('formal runner result is absent')
    result = json.loads(LOCAL_RESULT.read_text(encoding='ascii'))
    success = (
        completed.returncode == 0
        and result.get('status') == 'CALIBRATION_COMPLETE_THRESHOLD_CANDIDATE_READY'
        and result.get('fresh_replay_zero_of_256') is True
        and result.get('threshold_candidate_ready') is True
        and LOCAL_THRESHOLD.is_file()
    )
    stage = 'terminal_zip_publication'
    write_terminal_zip(success)
    terminal_published = True
    if not success:
        raise RuntimeError('calibration blocked; inspect the retained terminal ZIP')
except BaseException as error:
    root_key = ''
    hf_token = ''
    if runner_env is not None:
        runner_env.pop('CEG_WM_ROOT_KEY', None)
        runner_env.pop('HF_TOKEN', None)
    runner_env = None
    write_notebook_failure(stage, error)
    if not terminal_published and BLIND_TERMINAL_ZIP_WRITES == 0:
        write_terminal_zip(False)
        terminal_published = True
    raise
finally:
    root_key = ''
    hf_token = ''
    if runner_env is not None:
        runner_env.pop('CEG_WM_ROOT_KEY', None)
        runner_env.pop('HF_TOKEN', None)
    runner_env = None


In [ ]:
with zipfile.ZipFile(TERMINAL_ZIP, mode='r') as archive:
    members = set(archive.namelist())
    required = {
        'calibration_result.json', 'calibration_rows.json', 'fresh_replay_rows.json',
        'runner.stdout.txt', 'runner.stderr.txt',
    }
    if not required.issubset(members):
        raise RuntimeError('terminal ZIP members are incomplete')
    result = json.loads(archive.read('calibration_result.json').decode('ascii'))
    success = result.get('status') == 'CALIBRATION_COMPLETE_THRESHOLD_CANDIDATE_READY'
    threshold_present = 'blind_detection_v1_thresholds.json' in members
    if threshold_present != success:
        raise RuntimeError('terminal ZIP threshold/status consistency differs')
    print('CEGWM_BLIND_CALIBRATION_READBACK ' + json.dumps({
        'members': sorted(members),
        'producer_exact': result.get('producer_exact'),
        'status': result.get('status'),
    }, sort_keys=True))
